# ADMET-Engine v3.0 (*CardioPhys-3D*) — WP3 GPU Training Notebook (Colab Pro A100 / L4)

Trains the **SE(3)-equivariant PaiNN 3D protein-ligand model** (`CardioPhys3DPredictor`) on the WP1 multi-channel ChEMBL v37 dataset (**hERG**, **Cav1.2**, **Nav1.5**) with:
- Censored Gaussian **Tobit Negative Log-Likelihood** (`=`, `>`, `<`)
- **bfloat16** mixed precision on CUDA GPU
- **Boltzmann conformer sampling** at 310.15 K
- **Mondrian Conformal Prediction** (90% finite-sample coverage guarantee)
- End-to-end **WP3 → WP2 (ORd-CiPA ODE)** verification

In [ ]:
# @title 1. Sync Repo & Verify GPU
import os, torch
REPO_DIR = '/content/Admet-Engine'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/JakobFerschin/Admet-Engine.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!pip install -q rdkit torch-geometric joblib scikit-learn scipy requests
assert torch.cuda.is_available(), 'Please switch Colab Runtime to GPU (T4/L4/A100)!'
print('GPU:', torch.cuda.get_device_name(0), '| bfloat16:', torch.cuda.is_bf16_supported())

In [ ]:
# @title 2. Train SE(3)-Equivariant CardioPhys-3D with Tobit Censored NLL
!python src/training/train_wp3_3d.py \
    --csv data/processed/cardiophys_multichannel_v3.csv \
    --conformers data/processed/cardiophys_3d_conformers_v3.pt \
    --epochs 40 \
    --batch-size 64 \
    --hidden-dim 128 \
    --num-layers 3 \
    --lr 5e-4

In [ ]:
# @title 3. Inspect Held-Out Scaffold Metrics & Conformal Coverage
import json
metrics = json.loads(open('models/cardiophys_3d_v3_metrics.json').read())
print(json.dumps(metrics, indent=2))